# Lab 02 — Tokenizers, Alignment, and Real Model Outputs

**Tier 1 lab.** Executed during the build, every claim asserted. This lab needs a network
connection the first time it runs (to pull two small checkpoints and two extra tokenizers,
~1 GB total) and nothing else; it runs on CPU in a few minutes and unchanged inside a GPU
container. On the training box the identical code is how you'd smoke-test a real pair — swap
the model names for `Qwen3-8B` / `Qwen3-1.7B` and the assertions still hold, because
they check alignment mechanics, not model quality.

Lab 01 verified the objective on synthetic logits. This lab closes the remaining gap to reality:
**where teacher logits actually come from, and how they line up with the student's.** The models
are `SmolLM2-360M-Instruct` (teacher) and `SmolLM2-135M-Instruct` (student) — deliberately tiny,
but a *genuine* distillation pair: same tokenizer, same family, 2.7× capacity gap, the same
relationship a Qwen3-14B → 1.7B pair has on the training box.

The centerpiece is §3: our `shift_for_next_token` + masked cross-entropy reproduces Hugging
Face's own `model(labels=...).loss` to float precision. That single assertion grounds every mask
and shift in `kd_core` against the convention the entire ecosystem trains with. Get that right
and cached-logit training (Unit 04) is bookkeeping; get it wrong and you train on garbage that
converges anyway.

In [1]:
import sys, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

from kd_core import (
    kl_divergence, shift_for_next_token, completion_mask_from_prompt_lens,
    make_topk_cache, topk_forward_kl, topk_truncation_bias,
    bytes_per_token_cache, top1_agreement, mean_entropy, masked_mean,
)

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32          # tiny models: fp32 everywhere is fine and exact.
print(f"torch {torch.__version__} | device: {device}")

TEACHER = "HuggingFaceTB/SmolLM2-360M-Instruct"
STUDENT = "HuggingFaceTB/SmolLM2-135M-Instruct"
print(f"teacher: {TEACHER}\nstudent: {STUDENT}")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.13.0+cpu | device: cpu
teacher: HuggingFaceTB/SmolLM2-360M-Instruct
student: HuggingFaceTB/SmolLM2-135M-Instruct


## 1. Two tokenizers, one string: there is no shared coordinate system

A logit tensor is indexed `[batch, position, vocab_id]`. Both of the last two axes are
*tokenizer-defined*: position `t` means "after the first `t` tokens of *this* tokenizer's
segmentation", and the vocab axis means nothing without the vocab file. So before any talk of
teacher logits, establish the coordinate systems in play — and how different they are across
families you might be tempted to mix.

One subtlety the cell surfaces because it costs people real debugging hours:
`tokenizer.vocab_size`, `len(tokenizer)`, and the model's embedding rows are **three different
numbers**. Qwen2.5 reports a base `vocab_size` of 151,643, carries extra special tokens to
151,665, and pads its embedding matrix to 151,936 for kernel efficiency. Logit caches index the
*embedding* axis; anything you cache or compare must agree on which of the three you mean.

In [2]:
toks = {name: AutoTokenizer.from_pretrained(name)
        for name in ["gpt2", "Qwen/Qwen2.5-0.5B-Instruct", TEACHER]}

text = "With 273 GB/s of memory bandwidth, prefill is cheap and decode is not."

print(f"{'tokenizer':>28} {'vocab_size':>11} {'len(tok)':>9} {'tokens':>7}  first pieces")
seqs = {}
for name, tok in toks.items():
    ids = tok(text, add_special_tokens=False)["input_ids"]
    seqs[name] = ids
    pieces = tok.convert_ids_to_tokens(ids)[:6]
    print(f"{name:>28} {tok.vocab_size:>11} {len(tok):>9} {len(ids):>7}  {pieces}")
    assert tok.decode(ids) == text, f"{name}: round-trip decode must be lossless"

lens = {n: len(s) for n, s in seqs.items()}
assert len(set(lens.values())) > 1, "different tokenizers, different sequence lengths"
q = toks["Qwen/Qwen2.5-0.5B-Instruct"]
assert q.vocab_size != len(q), "Qwen: base vocab != vocab with special tokens"

# Fertility: tokens per UTF-8 byte. Lower = cheaper caching and faster prefill per char.
print(f"\nfertility (tokens/byte) on this string:")
for n, s in seqs.items():
    print(f"  {n:>28}: {len(s) / len(text.encode()):.3f}")
print("\nsame string, three incompatible coordinate systems — position-wise logit"
      "\nalignment across tokenizers is undefined before it is even difficult (see §6)")

                   tokenizer  vocab_size  len(tok)  tokens  first pieces
                        gpt2       50257     50257      18  ['With', 'Ġ273', 'ĠGB', '/', 's', 'Ġof']
  Qwen/Qwen2.5-0.5B-Instruct      151643    151665      20  ['With', 'Ġ', '2', '7', '3', 'ĠGB']
HuggingFaceTB/SmolLM2-360M-Instruct       49152     49152      21  ['With', 'Ġ', '2', '7', '3', 'ĠGB']

fertility (tokens/byte) on this string:
                          gpt2: 0.257
    Qwen/Qwen2.5-0.5B-Instruct: 0.286
  HuggingFaceTB/SmolLM2-360M-Instruct: 0.300

same string, three incompatible coordinate systems — position-wise logit
alignment across tokenizers is undefined before it is even difficult (see §6)


## 2. Chat templates decide what a "prompt" even is

Instruct models do not see your string; they see the chat template's rendering of it — special
tokens, role headers, a generation prompt. The prompt/completion boundary that
`completion_mask_from_prompt_lens` needs is therefore a fact about the *templated* token
sequence, and the only robust way to get it is to render the prompt side with
`add_generation_prompt=True` and measure its length in tokens.

One padding trap, flagged here because this lab then avoids it deliberately: SmolLM2-Instruct's
EOS is `<|im_end|>`, which legitimately *ends every completion*. If you also pad with the EOS
token and then exclude `pad_token_id` from the mask, you silently unmask the real EOS at the end
of every sequence — and a student never supervised on EOS is the model that never stops
(`kd_core.onpolicy_mask`'s docstring is about the same failure from the other side). We pad with
`<|endoftext|>`, which never occurs inside a templated conversation.

In [3]:
tok = toks[TEACHER]
PAD_ID = tok.convert_tokens_to_ids("<|endoftext|>")
EOS_ID = tok.eos_token_id
assert PAD_ID != EOS_ID, "pad must differ from EOS or the mask eats the real EOS"

pairs = [
    ("What is 17 * 23?", "17 * 23 = 391."),
    ("Name the largest planet in the solar system.", "Jupiter."),
    ("Write one sentence about autumn.", "The leaves turn amber and the air smells of rain."),
    ("In Python, how do I reverse a list in place?", "Call `my_list.reverse()`."),
]

prompt_ids, full_ids = [], []
for user_msg, completion in pairs:
    p = tok.apply_chat_template([{"role": "user", "content": user_msg}],
                                add_generation_prompt=True, tokenize=True,
                                return_dict=False)
    c = tok(completion, add_special_tokens=False)["input_ids"] + [EOS_ID]
    prompt_ids.append(p)
    full_ids.append(p + c)

T_max = max(len(x) for x in full_ids)
input_ids = torch.full((len(pairs), T_max), PAD_ID)
for i, x in enumerate(full_ids):
    input_ids[i, :len(x)] = torch.tensor(x)
prompt_lens = [len(p) for p in prompt_ids]

mask = completion_mask_from_prompt_lens(input_ids, prompt_lens, pad_token_id=PAD_ID)

print("rendered prompt for pair 0:")
print(repr(tok.decode(prompt_ids[0])))
print(f"\nbatch [{input_ids.shape[0]}, {input_ids.shape[1]}], prompt lens {prompt_lens}")
for i, (p, f) in enumerate(zip(prompt_ids, full_ids)):
    n_comp = len(f) - len(p)
    assert int(mask[i].sum()) == n_comp, f"row {i}: mask must cover exactly the completion"
    assert not mask[i, :len(p)].any(), f"row {i}: no prompt token may be supervised"
    assert mask[i, len(f) - 1], f"row {i}: the EOS token must be supervised"
print("mask covers completion + EOS, never prompt, never padding — verified")

rendered prompt for pair 0:
'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is 17 * 23?<|im_end|>\n<|im_start|>assistant\n'

batch [4, 53], prompt lens [40, 39, 36, 42]
mask covers completion + EOS, never prompt, never padding — verified


## 3. The assertion that grounds everything: reproducing HF's loss by hand

Hugging Face models shift labels *internally*: `model(input_ids, labels=labels)` scores logits
at position `t` against the label at `t+1` and averages over positions whose label is not
`-100`. `kd_core` keeps the shift *external* and explicit, because distillation needs the same
alignment applied to a *teacher* tensor that HF's convenience path never sees.

If the two conventions agree to float precision on real models and a ragged batch — prompts
masked, padding masked, EOS supervised — then every masked divergence in this course sits on the
same coordinate system the ecosystem trains with. This is the single most load-bearing cell in
the Tier 1 track.

In [4]:
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, dtype=DTYPE).to(device).eval()
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=DTYPE).to(device).eval()
input_ids_d, mask_d = input_ids.to(device), mask.to(device)

# HF's path: -100 everywhere we do not supervise.
labels = input_ids_d.clone()
labels[~mask_d] = -100

with torch.no_grad():
    out_s = student(input_ids_d, labels=labels)
    s_logits = out_s.logits
    t_logits = teacher(input_ids_d).logits
hf_loss = float(out_s.loss)

# kd_core's path: explicit shift, explicit mask, per-token masked mean.
s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, mask_d)
tokens_sh = input_ids_d[:, 1:]
lp = F.log_softmax(s_sh, dim=-1)
nll = -lp.gather(-1, tokens_sh.unsqueeze(-1)).squeeze(-1)
our_loss = float(masked_mean(nll, m_sh))

print(f"HF model(labels=...).loss : {hf_loss:.6f}")
print(f"shift + mask + masked_mean: {our_loss:.6f}")
assert abs(hf_loss - our_loss) < 1e-4, "kd_core alignment must reproduce HF's internal shift"
assert int(m_sh.sum()) == int(mask.sum()), "shift may not change the number of supervised positions"
print("kd_core's external shift reproduces HF's internal one to float precision")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:14<1:11:25, 14.83s/it]

Loading weights:   9%|▉         | 27/290 [00:16<01:57,  2.24it/s] 

Loading weights:  10%|█         | 30/290 [00:18<02:08,  2.02it/s]

Loading weights:  12%|█▏        | 36/290 [00:20<01:45,  2.41it/s]

...

Loading weights:  88%|████████▊ | 238/272 [00:23<00:03,  8.83it/s]

Loading weights:  93%|█████████▎| 254/272 [00:23<00:01, 14.74it/s]

Loading weights:  97%|█████████▋| 264/272 [00:23<00:00, 19.12it/s]

Loading weights: 100%|██████████| 272/272 [00:23<00:00, 11.46it/s]

HF model(labels=...).loss : 2.275541
shift + mask + masked_mean: 2.275541
kd_core's external shift reproduces HF's internal one to float precision


## 4. Teacher and student, measured like a distillation run

With alignment trusted, measure the pair the way Unit 11 says to: forward KL on supervised
positions, top-1 agreement (the honest headline number), and mean entropy — not perplexity
against gold text.

The interesting assertion is the control: agreement between the real teacher and the real
student must beat agreement between the teacher and a **randomly initialised** copy of the
student architecture by a wide margin. That control is cheap, and running it against your own
pair before a Tier 2 run tells you whether the numbers you are about to improve are even
measuring a relationship. The per-position table shows *where* the divergence lives — peaked
positions where both agree contribute almost nothing; the KL concentrates on genuinely open
positions.

In [5]:
fwd_kl = kl_divergence(s_sh, t_sh, m_sh, direction="forward", scale_by_T2=False)
agree  = top1_agreement(s_sh, t_sh, m_sh)
H_t, H_s = mean_entropy(t_sh, m_sh), mean_entropy(s_sh, m_sh)

# Control: an untrained model with the student's architecture.
noise = AutoModelForCausalLM.from_config(
    AutoConfig.from_pretrained(STUDENT)).to(device).to(DTYPE).eval()
with torch.no_grad():
    n_logits = noise(input_ids_d).logits
n_sh, _, _ = shift_for_next_token(n_logits, t_logits, mask_d)
agree_noise = top1_agreement(n_sh, t_sh, m_sh)

print(f"forward KL(teacher||student) : {float(fwd_kl):.4f} nats/token")
print(f"top-1 agreement              : {agree:.3f}   (random-init control: {agree_noise:.3f})")
print(f"mean entropy  teacher {H_t:.3f}  student {H_s:.3f}  nats")

assert float(fwd_kl) > 0
assert 0.3 < agree < 1.0, "a same-family pair should agree often but not always"
assert agree > agree_noise + 0.3, "trained student must beat the random-init control by a margin"

# Where the divergence lives: per-position view of pair 2 (the open-ended one).
i = 2
kl_pos = (F.softmax(t_sh[i], -1) * (F.log_softmax(t_sh[i], -1)
          - F.log_softmax(s_sh[i], -1))).sum(-1)
print(f"\nper-position, pair 2 ({pairs[i][0]!r}):")
print(f"{'position':>32} {'teacher top-1':>14} {'p(top1)':>8} {'KL':>7}")
for t in torch.nonzero(m_sh[i]).squeeze(-1)[:10].tolist():
    p_t = F.softmax(t_sh[i, t], -1)
    top_p, top_i = p_t.max(-1)
    ctx = tok.decode(input_ids[i, max(0, t - 3 + 1): t + 1].tolist())
    print(f"{ctx[-32:]!r:>32} {tok.decode([top_i]).strip()!r:>14} {float(top_p):>8.3f} "
          f"{float(kl_pos[t]):>7.3f}")

forward KL(teacher||student) : 0.5651 nats/token
top-1 agreement              : 0.615   (random-init control: 0.000)
mean entropy  teacher 1.293  student 1.515  nats

per-position, pair 2 ('Write one sentence about autumn.'):
                        position  teacher top-1  p(top1)      KL
                   'assistant\n'          'Aut'    0.443   0.289
                   'istant\nThe'       'season'    0.523   1.378
                  '\nThe leaves'           'of'    0.372   0.584
               'The leaves turn'    'brilliant'    0.216   0.631
            ' leaves turn amber'          'and'    0.518   0.393
               ' turn amber and'         'gold'    0.545   1.336
                ' amber and the'          'air'    0.227   1.413
                  ' and the air'        'cools'    0.638   1.598
               ' the air smells'           'of'    0.715   0.779
                ' air smells of'       'fallen'    0.107   1.963


## 5. Top-k caching against real teacher distributions

Lab 01 §6 measured top-k truncation bias on synthetic logits, where "peaked" and "flat" were
knobs. Here the peakedness is whatever a real instruct model actually produces on real
completions — and the real data promptly complicates the synthetic story, which is the point.
On this pair, the two estimators are biased in **opposite directions**: renormalising over the
kept entries *overstates* the KL (it deletes the tail where teacher and student roughly agree,
leaving the contested head overweighted), while the tail bucket *understates* it (one aggregate
bucket cannot see disagreement in how the tail mass is distributed within it). Both converge to
the dense value as `k` grows, one from each side — the build originally asserted "the tail
bucket always wins", and the `k=4` row refused. Which estimator is closer at *your* `k` on
*your* corpus is a measurement, not a theorem. That is the workflow this section leaves you
with: run `topk_truncation_bias` on a sample of the actual corpus before committing to a cache
design — the pre-flight step Unit 04 requires.

Then the arithmetic that makes this a *hardware* decision. Caching is pure prefill — nobody
decodes — which the README's workload table prices as "cheap, pay once". The cell prices it
concretely: storage per million tokens at the chosen `k`, and wall-clock at a measured prefill
rate for a 20B-class teacher on 273 GB/s-class hardware, for the real vocabularies from §1.

In [6]:
rows = topk_truncation_bias(s_sh, t_sh, m_sh, ks=(1, 4, 16, 64, 128))
dense = rows[0]["dense_kl"]
print(f"{'k':>4} {'mass kept':>10} {'renorm KL':>10} {'tail KL':>9}   dense KL = {dense:.4f}")
for r in rows:
    print(f"{r['k']:>4} {r['mean_mass_covered']:>10.4f} {r['renorm_kl']:>10.4f} "
          f"{r['tail_bucket_kl']:>9.4f}")

renorms = [r["renorm_kl"] for r in rows]
tails   = [r["tail_bucket_kl"] for r in rows]
# Opposite-direction biases, each converging monotonically to dense as k grows.
assert all(r >= dense - 1e-6 for r in renorms), "renormalising overstates KL on this teacher"
assert all(t <= dense + 1e-6 for t in tails), "the tail bucket understates KL on this teacher"
assert all(a >= b - 1e-6 for a, b in zip(renorms, renorms[1:])), "renorm bias shrinks with k"
assert all(a <= b + 1e-6 for a, b in zip(tails, tails[1:])), "tail bias shrinks with k"
k128 = rows[-1]
assert abs(k128["renorm_rel_err"]) < 0.10 and abs(k128["tail_rel_err"]) < 0.10, \
    "by k=128 both estimators should sit within 10% of dense"
k64 = next(r for r in rows if r["k"] == 64)
assert k64["mean_mass_covered"] > 0.98, "a real instruct teacher is peaked; k=64 should cover >98%"

print("\nstorage and prefill cost per 1M cached tokens:")
PREFILL_TOKS_PER_S = 2053   # measured, 20B-class teacher in MXFP4 on 273 GB/s hardware
for name, V in [("SmolLM2 (49,152)", 49152), ("Llama 3 (128,256)", 128256),
                ("Qwen3 (151,936 padded)", 151936)]:
    s = bytes_per_token_cache(V, k=64)
    print(f"  {name:>24}: dense {s['dense_gb_per_1M_tokens']:.1f} GB -> top-64 "
          f"{s['topk_gb_per_1M_tokens']:.3f} GB ({s['compression']:.0f}x), "
          f"~{1e6 / PREFILL_TOKS_PER_S / 60:.1f} min prefill at {PREFILL_TOKS_PER_S} tok/s")
s = bytes_per_token_cache(151936, k=64)
assert s["dense_gb_per_1M_tokens"] > 100 * s["topk_gb_per_1M_tokens"]

   k  mass kept  renorm KL   tail KL   dense KL = 0.5651
   1     0.6744     1.1326    0.2738
   4     0.8761     0.7172    0.3971
  16     0.9527     0.6169    0.4868
  64     0.9829     0.5801    0.5343
 128     0.9900     0.5730    0.5454

storage and prefill cost per 1M cached tokens:
          SmolLM2 (49,152): dense 98.3 GB -> top-64 0.386 GB (255x), ~8.1 min prefill at 2053 tok/s
         Llama 3 (128,256): dense 256.5 GB -> top-64 0.386 GB (665x), ~8.1 min prefill at 2053 tok/s
    Qwen3 (151,936 padded): dense 303.9 GB -> top-64 0.386 GB (787x), ~8.1 min prefill at 2053 tok/s


## 6. Cross-tokenizer alignment: measuring how bad it is

Unit 10 covers distilling across tokenizers (ULD's sorted-logit matching, GOLD). This section
only establishes the *problem*, with a number: using each tokenizer's byte offsets, count how
many token boundaries GPT-2 and SmolLM2 even place at the same byte position of the same string.

Positions that don't share a boundary can't be compared at all; positions that do share one
still index incompatible vocabularies, so the per-position KL is between distributions over
*different outcome spaces* — undefined, not merely noisy. Everything principled in Unit 10
starts from accepting that, and works with quantities that survive re-tokenization: bytes,
strings, or sorted probability values.

In [7]:
sample = ("Distillation transfers the teacher's distribution, not its weights. "
          "273 GB/s is the budget; design the pipeline around prefill.")

def boundaries(tok, text):
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    return [e for _, e in enc["offset_mapping"]], len(enc["input_ids"])

b_gpt2, n_gpt2 = boundaries(toks["gpt2"], sample)
b_smol, n_smol = boundaries(toks[TEACHER], sample)
shared = set(b_gpt2) & set(b_smol)
frac = len(shared) / max(len(set(b_gpt2)), len(set(b_smol)))

print(f"gpt2: {n_gpt2} tokens | SmolLM2: {n_smol} tokens | "
      f"shared boundaries: {len(shared)} ({frac:.0%})")
assert n_gpt2 != n_smol or b_gpt2 != b_smol, "the segmentations must differ"
assert frac < 1.0, "not all boundaries coincide -> no global position-wise alignment"
assert len(shared) > 0, "some boundaries coincide (word edges) -> partial alignment exists"
print("position-wise alignment across these tokenizers is partial at best;"
      "\nUnit 10's methods exist because of this measurement")

gpt2: 27 tokens | SmolLM2: 30 tokens | shared boundaries: 27 (90%)
position-wise alignment across these tokenizers is partial at best;
Unit 10's methods exist because of this measurement


## Exercises

1. **Run the §3 assertion against a training-scale pair.** On the training box, swap in
   `Qwen/Qwen3-8B` and `Qwen/Qwen3-1.7B` (bf16, `device_map="auto"`), and confirm the HF-loss
   cross-check still passes. Note what you had to change (dtype tolerance) and what you did not
   (any line of alignment code).
2. **Find your k on your corpus.** Replace the four toy pairs with 200 prompts from the corpus
   you actually intend to distill on, rerun §5, and commit to a `k` with a written justification:
   mass covered, tail error, GB on disk, prefill minutes.
3. **Break the padding on purpose.** Re-pad §2 with `pad_token_id = eos_token_id` and watch
   which assertion fails. Explain the downstream symptom in a trained student.
4. **Fertility as a cost model.** Tokenize 1,000 lines of your corpus with all three tokenizers
   and compute total tokens. A teacher whose tokenizer is 15% more fertile pays 15% more
   prefill, cache storage, *and* per-token decode on every rollout it ever scores. Which of the
   §1 tokenizers would you want owning your corpus?
5. **Sorted logits survive re-tokenization.** For a boundary position shared by both §6
   tokenizers, take each model's next-token distribution, sort the probability vectors, and
   compare the sorted tails. This is the first step of ULD, and you have already written it.